# Lectora — API Flow Debug Notebook
Run cells top-to-bottom, or jump to any cell by setting the required IDs manually.

**Prereq:** `uvicorn lectora_backend.dev_app:app --reload --port 8000`

In [ ]:
import json, time, requests
from pathlib import Path

BASE_URL = "http://localhost:8000"

def call(method, path, payload=None, files=None, form=None, timeout=300):
    url = f"{BASE_URL}{path}"
    print(f">>> {method.upper()} {url}")
    if payload:
        print(json.dumps(payload, indent=2))
    t0 = time.time()
    r  = requests.request(method, url, json=payload, files=files, data=form, timeout=timeout)
    ms = int((time.time() - t0) * 1000)
    print(f"<<< {r.status_code}  ({ms} ms)")
    try:
        print(json.dumps(r.json(), indent=2))
    except Exception:
        print(r.text[:500])
    return r

print("ready")

---
## Step 1 — Health check

In [ ]:
r = call("GET", "/health/")

---
## Step 2 — Upload study guide

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
FILE_PATH   = "./sample_study_guide.docx"   # path to your DOCX or PDF
COURSE_TOPIC = "Insurance Fundamentals"      # becomes the upload folder name
# ─────────────────────────────────────────────────────────────────────────────

with open(FILE_PATH, "rb") as f:
    r = call(
        "POST", "/documents/upload",
        files={"file": (Path(FILE_PATH).name, f, "application/octet-stream")},
        form={"courseTopic": COURSE_TOPIC},
    )

STUDY_GUIDE_BLOB = r.json().get("blobPath")
print("\nSTUDY_GUIDE_BLOB =", STUDY_GUIDE_BLOB)

---
## Step 3 — Generate Training Outline (A0)

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
payload = {
    # mandatory
    "blobPaths":               [STUDY_GUIDE_BLOB],   # from Step 2, or paste manually
    "audience":                "Licensed Insurance Agents",

    # course identity
    "courseTitle":             "Employer-Provided Health Plans",
    "courseDescription":       "",
    "courseTypeHint":          "insurance_ce",

    # duration & difficulty (drives dynamic TO prompt)
    "difficulty":              "intermediate",
    "difficultyLevel":         "intermediate",
    "durationHours":           3.0,
    "calculatedWordCount":     21600,              # durationHours × 9000 / 1.25

    # wizard content-direction fields
    "experienceLevel":         "",                 # e.g. "beginner", "advanced"
    "learnerOutcomes":         "",
    "audienceNotes":           "",
    "learningObjectives":      [],
    "tone":                    "",
    "depth":                   "",
    "emphasis":                "",
    "avoid":                   "",
    "includeScenarios":        True,
    "includeKnowledgeChecks":  True,
    "preferredChapters":       None,               # int or None
    "lessonStyle":             "",

    # source analysis results (from Step 14, optional)
    "sourceAnalyses":          [],

    # required topics from wizard
    "requiredTopics":          [],

    # custom prompt override (optional)
    # "customToPrompt":        "...",

    # pre-built TO doc (skip generation entirely, optional)
    # "toDocBlobPath":         "...",
}
# ─────────────────────────────────────────────────────────────────────────────

# Remove None values before sending
payload = {k: v for k, v in payload.items() if v is not None}

r = call("POST", "/documents/generate-to", payload)

TO_JOB_ID = r.json().get("jobId")
print("\nTO_JOB_ID =", TO_JOB_ID)

---
## Step 4 — Poll A0 until complete

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
# TO_JOB_ID = "paste-job-id-here"   # uncomment to skip Step 3
# ─────────────────────────────────────────────────────────────────────────────

while True:
    r    = requests.get(f"{BASE_URL}/documents/generate-to/jobs/{TO_JOB_ID}", timeout=30)
    body = r.json()
    st   = body.get("status")
    print(f"status: {st}")
    if st in ("completed", "failed", "cancelled"):
        break
    time.sleep(3)

print(json.dumps(body, indent=2))
TO_BLOB_PATH = body.get("toBlobPath")
TO_OUTLINE   = body.get("to")    # full TO JSON — passed as toOverride in Step 5
print("\nTO_BLOB_PATH =", TO_BLOB_PATH)

---
## Step 5 — Create pipeline job

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
payload = {
    "courseTitle": "Employer-Provided Health Plans",
    "courseType":  "insurance_ce",              # insurance_ce | iarce | firm_element
    "difficulty":  "intermediate",
    "audience":    "Licensed Insurance Agents",

    # mandatory input docs
    "inputs": {
        "studyGuide":   {"blobPath": STUDY_GUIDE_BLOB},   # from Step 2
        "timedOutline": {"blobPath": TO_BLOB_PATH},       # from Step 4 (remove if no TO)
    },

    # full TO JSON from the three-panel editor (user edits flow back here)
    "toOverride": TO_OUTLINE if "TO_OUTLINE" in dir() else None,

    # optional: additional source files beyond the study guide
    "sourceFileSpecs": [
        # {"blobPath": "...", "extractHint": "focus on compliance sections", "importance": "high"}
    ],

    # optional free-text instructions for A2
    # "specialInstructions": "...",

    # wizard fields forwarded to A2 for dynamic prompt construction
    "courseConfig": {
        "courseTitle":             "Employer-Provided Health Plans",
        "courseDescription":       "",
        "experienceLevel":         "",
        "learnerOutcomes":         "",
        "audienceNotes":           "Licensed Insurance Agents",
        "learningObjectives":      [],
        "tone":                    "",
        "depth":                   "",
        "emphasis":                "",
        "avoid":                   "",
        "includeScenarios":        True,
        "includeKnowledgeChecks":  True,
    },
}
# ─────────────────────────────────────────────────────────────────────────────

# Remove None values and empty dicts at top level
payload = {k: v for k, v in payload.items() if v is not None}
if not payload.get("sourceFileSpecs"):
    del payload["sourceFileSpecs"]

r = call("POST", "/jobs", payload)

JOB_ID = r.json().get("jobId")
print("\nJOB_ID =", JOB_ID)

---
## Step 6 — Poll pipeline until complete

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
# JOB_ID = "paste-job-id-here"   # uncomment to skip Step 5
# ─────────────────────────────────────────────────────────────────────────────

seen_logs = set()
last_stages = {}

while True:
    r    = requests.get(f"{BASE_URL}/jobs/{JOB_ID}", timeout=30)
    body = r.json()
    st   = body.get("status")

    for log in body.get("logs", []):
        lid = log.get("id") or id(log)
        if lid not in seen_logs:
            seen_logs.add(lid)
            print(f"  [{log.get('stageId','?'):12s}][{log.get('level','info').upper():7s}] {log.get('message','')}")

    for stg in body.get("stages", []):
        sid  = stg.get("stage") or stg.get("stageId")
        sst  = stg.get("status")
        if last_stages.get(sid) != sst:
            print(f"  STAGE {sid}: {sst}  outcome={stg.get('outcome','')}")
            last_stages[sid] = sst

    if st in ("COMPLETED", "FAILED", "CANCELLED"):
        print(f"\nFinal status: {st}")
        break
    time.sleep(3)

---
## Step 7 — Get course content

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
# JOB_ID = "paste-job-id-here"   # uncomment to use a specific job
# ─────────────────────────────────────────────────────────────────────────────

r = call("GET", f"/jobs/{JOB_ID}/course")

sections = r.json().get("sections", [])
print("\nSections:")
for s in sections:
    print(f"  {s.get('id'):<45}  {s.get('sectionType'):<25}  {s.get('wordCount',0):>5} words")

# Pick first content section for use in Steps 8–10
SECTION = next((s for s in sections if s.get("sectionType") == "content"), sections[0] if sections else {})
SECTION_ID      = SECTION.get("id", "")
SECTION_CONTENT = SECTION.get("content", "")
print("\nSECTION_ID =", SECTION_ID)

---
## Step 8 — AI operation (rewrite / expand / simplify / summarize / improve_tone / regenerate)

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
payload = {
    "operation":  "rewrite",       # rewrite | expand | simplify | summarize | improve_tone | regenerate
    "sectionId":  SECTION_ID,      # from Step 7, or paste a section id
    "content":    SECTION_CONTENT, # current section text sent by FE
    "userPrompt": "Make it more concise and add a real-world example.",  # required for rewrite/improve_tone, ignored by others
}
# ─────────────────────────────────────────────────────────────────────────────

r = call("POST", f"/jobs/{JOB_ID}/ai", payload)

---
## Step 9 — Save section content

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
SECTION_ID_TO_SAVE = SECTION_ID   # or paste any section id from Step 7
payload = {
    "content":     SECTION_CONTENT,  # edited text from the FE editor
    "sectionType": "content",        # content | overview | conclusion | learning-objectives
}
# ─────────────────────────────────────────────────────────────────────────────

r = call("PATCH", f"/jobs/{JOB_ID}/sections/{SECTION_ID_TO_SAVE}", payload)

---
## Step 10 — Download DOCX

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
SAVE_AS = f"output_{JOB_ID[:8]}.docx"
# ─────────────────────────────────────────────────────────────────────────────

r = requests.get(f"{BASE_URL}/jobs/{JOB_ID}/artifacts/download", timeout=120)
print(f"<<< {r.status_code}  content-type: {r.headers.get('content-type')}")

if "json" in r.headers.get("content-type", ""):
    print(json.dumps(r.json(), indent=2))   # signed URL (production mode)
else:
    Path(SAVE_AS).write_bytes(r.content)
    print(f"Saved → {Path(SAVE_AS).resolve()}  ({len(r.content)//1024} KB)")

---
## Step 11 — List artifacts

In [ ]:
r = call("GET", f"/jobs/{JOB_ID}/artifacts")

---
## Step 12 — Poll job status (one-shot check)

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
# JOB_ID = "paste-job-id-here"
# ─────────────────────────────────────────────────────────────────────────────

r = call("GET", f"/jobs/{JOB_ID}")

---
## Step 13 — Retry failed job

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
# JOB_ID = "paste-job-id-here"
payload = {
    "fromStage": "A0",   # stage to restart from: A0 | A1 | S1 | A2
}
# ─────────────────────────────────────────────────────────────────────────────

r = call("POST", f"/jobs/{JOB_ID}/retry", payload)

---
## Step 14 — Analyze source document

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
payload = {
    "blobPath":   STUDY_GUIDE_BLOB,   # from Step 2, or paste manually
    "sourceRole": "primary_source",   # primary_source | supporting_source | reference_only
    "importance": "high",             # high | medium | low
}
# ─────────────────────────────────────────────────────────────────────────────

r = call("POST", "/documents/analyze-source", payload)

SOURCE_ANALYSIS = r.json() if r.ok else {}

---
## Step 15 — Generate learning objectives by AI

In [ ]:
# ── PAYLOAD ──────────────────────────────────────────────────────────────────
payload = {
    "courseTitle":            "Employer-Provided Health Plans",
    "courseDescription":      "",
    "courseType":             "insurance_ce",
    "courseDuration":         "3 hours",
    "targetAudience":         "Licensed Insurance Agents",
    "skillLevel":             "intermediate",
    "desiredOutcomes":        "",
    "certificationFocus":     "",
    "additionalInstructions": "",
    "sourceMaterials":        [STUDY_GUIDE_BLOB],               # blob paths of all uploaded docs
    "sourceAnalyses":         [SOURCE_ANALYSIS] if SOURCE_ANALYSIS else [],  # from Step 14 (optional)
    "requiredTopics":         [],
}
# ─────────────────────────────────────────────────────────────────────────────

r = call("POST", "/documents/generate-learning-objectives", payload)

if r.ok:
    for i, lo in enumerate(r.json().get("learningObjectives", []), 1):
        print(f"  {i}. {lo}")